# Lab 13 — 3D Classification, Detection and Segmentation
Synthetic 3D medical volumes demonstrate the three tasks. Educational use only.

In [ ]:
import numpy as np,torch,torch.nn as nn,torch.optim as optim
from torch.utils.data import Dataset,DataLoader
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); rng=np.random.default_rng(42)

In [ ]:
class VolDS(Dataset):
 def __init__(self,n=80,s=24): self.items=[]; self.s=s; [self.items.append(self.make()) for _ in range(n)]
 def make(self):
  s=self.s; v=rng.normal(0,.05,(s,s,s)).astype('float32'); m=np.zeros_like(v); positive=rng.random()>.35
  if positive:
   c=rng.integers(6,s-6,3); r=int(rng.integers(2,5)); zz,yy,xx=np.ogrid[:s,:s,:s]; mask=(zz-c[0])**2+(yy-c[1])**2+(xx-c[2])**2<=r*r; v[mask]+=1; m[mask]=1; lo=np.maximum(c-r,0)/s; hi=np.minimum(c+r+1,s)/s; box=np.r_[lo,hi].astype('float32')
  else: box=np.zeros(6,dtype='float32')
  return torch.tensor(v[None]),torch.tensor(int(positive)),torch.tensor(box),torch.tensor(m[None])
 def __len__(self): return len(self.items)
 def __getitem__(self,i): return self.items[i]
train=VolDS(80); test=VolDS(20); tr=DataLoader(train,batch_size=4,shuffle=True); te=DataLoader(test,batch_size=4)

In [ ]:
class C3D(nn.Module):
 def __init__(self,out=2): super().__init__(); self.f=nn.Sequential(nn.Conv3d(1,4,3,padding=1),nn.ReLU(),nn.MaxPool3d(2),nn.Conv3d(4,8,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool3d(1)); self.fc=nn.Linear(8,out)
 def forward(self,x): return self.fc(self.f(x).flatten(1))
cls=C3D().to(device); opt=optim.Adam(cls.parameters(),1e-3); loss_fn=nn.CrossEntropyLoss()
for e in range(3):
 for x,y,b,m in tr:
  x=x.to(device); y=y.to(device); opt.zero_grad(); loss=loss_fn(cls(x),y); loss.backward(); opt.step()
print('3D classification trained')

In [ ]:
class Detector(nn.Module):
 def __init__(self): super().__init__(); self.f=C3D(out=6)
 def forward(self,x): return torch.sigmoid(self.f(x))
det=Detector().to(device); opt=optim.Adam(det.parameters(),1e-3); mse=nn.MSELoss()
for e in range(3):
 for x,y,b,m in tr:
  mask=y.bool()
  if mask.any():
   x=x[mask].to(device); b=b[mask].to(device); opt.zero_grad(); loss=mse(det(x),b); loss.backward(); opt.step()
print('3D detector trained')

In [ ]:
class Seg3D(nn.Module):
 def __init__(self): super().__init__(); self.net=nn.Sequential(nn.Conv3d(1,4,3,padding=1),nn.ReLU(),nn.Conv3d(4,4,3,padding=1),nn.ReLU(),nn.Conv3d(4,1,1))
 def forward(self,x): return self.net(x)
seg=Seg3D().to(device); opt=optim.Adam(seg.parameters(),1e-3); bce=nn.BCEWithLogitsLoss()
for e in range(3):
 for x,y,b,m in tr:
  x=x.to(device); m=m.to(device); opt.zero_grad(); loss=bce(seg(x),m); loss.backward(); opt.step()
print('3D segmentation trained')

In [ ]:
cls.eval(); correct=total=0; dices=[]
with torch.no_grad():
 for x,y,b,m in te:
  p=cls(x.to(device)).argmax(1).cpu(); correct+=(p==y).sum().item(); total+=len(y); s=(torch.sigmoid(seg(x.to(device)))>.5).cpu(); inter=(s*m).sum((1,2,3,4)); dices.extend(((2*inter+1)/(s.sum((1,2,3,4))+m.sum((1,2,3,4))+1)).numpy())
print('Classification accuracy',correct/total); print('Mean segmentation Dice',float(np.mean(dices)))